## 제출 정보

- 이름: 예민호

- GitHub ID: yegamanse-cpu

- 작성일: 2026.11.17

- 최종 제출 URL: https://github.com/yegamanse-cpu/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb


In [12]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-course
전처리 데이터 폴더: C:\dev\llm-data-analysis-course\data\processed
보고서 폴더: C:\dev\llm-data-analysis-course\reports


- scripts/prepare_ch10_data.py 파일 생성

## 1. 분류 문제 정의

- 타깃: is_cancelled 

- 0의 의미: completed 

- 1의 의미: cancelled 

- 제외한 상태: refunded 및 기타 상태 

- 예측 단위: 주문 1건 

- 예측 시점: 주문접수 시점

- 예측 시점에 알 수 있는 정보: 주문일자, 결제수단, 나이·성별·도시

- 예측 시점에 알 수 없는 정보: order_status, is_cancelled 등

### 나의 해석과 판단

- 주문접수 시 취소여부는 알 수 없으므로 접수시점으로 설정

In [13]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())

customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


In [9]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary

,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature

- age, item_count, total_quantity, order_amount, order_month, order_dayofweek

### 사용한 범주형 Feature

- gender, city, payment_method

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

### 나의 해석과 판단

- order_status을 사용하면 정답을 알려주는 셈이기 떄문에 제외필요


In [14]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
    build_feature_audit,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)
feature_audit = build_feature_audit(numeric_features, categorical_features)

print("모델링 데이터:", model_data.shape)
print("숫자형 feature:", numeric_features)
print("범주형 feature:", categorical_features)
display(target_dist)
display(merge_checks)
display(feature_audit)

ValueError: signup_date가 order_date보다 늦은 주문이 있습니다: 45건

- "signup_date가 order_date보다 늦은 주문이 있습니다: 45건" 오류발생

- 해당 데이터는 제외가 필요할 것으로 판단, LLM을 활용하여 코드생성 및 실행

In [15]:
# [데이터 품질 게이트] signup_date가 order_date보다 늦은(가입 전) 주문 제외
# - build_classification_dataset는 days_since_signup < 0 이 있으면 ValueError를 던짐
# - 미매칭·이상치를 0으로 메우지 않고, 원인 확인 전까지 모델링 입력에서 제외
signup_lookup = customers[["customer_id", "signup_date"]].copy()
order_dates = orders.merge(signup_lookup, on="customer_id", how="left")
order_dates["order_date"] = pd.to_datetime(order_dates["order_date"])
order_dates["signup_date"] = pd.to_datetime(order_dates["signup_date"])

pre_signup_mask = order_dates["order_date"] < order_dates["signup_date"]
pre_signup_order_ids = order_dates.loc[pre_signup_mask, "order_id"]

print("가입 전 주문(제외 대상):", pre_signup_order_ids.nunique(), "건")
print("제외 전 orders:", len(orders), "| order_items:", len(order_items))

orders = orders[~orders["order_id"].isin(pre_signup_order_ids)].copy()
order_items = order_items[order_items["order_id"].isin(orders["order_id"])].copy()

print("제외 후 orders:", len(orders), "| order_items:", len(order_items))

가입 전 주문(제외 대상): 56 건
제외 전 orders: 300 | order_items: 764
제외 후 orders: 244 | order_items: 624


- 앞선 가이드 코드에선 제외대상 주문건이 45건이었으나 LLM이 만든 코드는 56건을 제외대상으로 계산

- 이유를 살펴보니 가이드코드의 제외대상 45건은 completed와 cancelled를 합친 수

- 신규코드가 제시한 제외 대상은 전체건수에서 refunded도 포함되어 11건을 더한 총 56건으로 계산됨

- refunded는 서두에 데이터에서 제외하기로 했으니 최종적으로 completed와 cancelled의 이상치는 정상적으로 제외되는 것으로 판단

In [16]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
    build_feature_audit,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)
feature_audit = build_feature_audit(numeric_features, categorical_features)

print("모델링 데이터:", model_data.shape)
print("숫자형 feature:", numeric_features)
print("범주형 feature:", categorical_features)
display(target_dist)
display(merge_checks)
display(feature_audit)

모델링 데이터: (203, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,149,0.734
1,1,cancelled,54,0.266


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,203,203,True,0,PASS
1,customer_id → customers,203,203,True,0,PASS


,column,selected,role,reason
0,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,item_count,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,total_quantity,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,order_amount,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,days_since_signup,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
7,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
8,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
9,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정


In [17]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')

사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


- line_total 불일치 확인을 위한 신규 코드 추가(LLM활용)

In [19]:
# line_total = quantity × unit_price 확인
mismatch = (
    order_items["line_total"] != order_items["quantity"] * order_items["unit_price"]
).sum()
print("order_items 행 수:", len(order_items))
print("line_total 불일치:", int(mismatch))

order_items 행 수: 624
line_total 불일치: 0


## 3. 주문 단위 특징과 병합 검증

### 계산 기준

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0 

- 주문 특징 order_id 중복 건수: 0 

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 203 | 203 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 203 | 203 | 0 | PASS |

### 결과 관찰

- 이상 데이터를 제외한 총 데이터는 203건이며 line_total 불일치 0, 주문 특징 중복 0

### 나의 해석과 판단

- 제외코드 생성 등 막히는 부분을 LLM을 통해 해결했고 불일치 건수나 중복건수가 0인것으로 보아 데이터가 이상없는것으로 판단됨


## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 149 | 73.4% |
| 1 | cancelled | 54 | 26.6% |

![클래스 비율](images/01.png)

### 결과 관찰

- completed 149(73.4%) / cancelled 54(26.6%)

### 나의 해석과 판단

- 완료주문과 취소주문건수가 약 3배 가까이 차이가 나서 accuracy가 정확하지 않을 수 있을거라 판단


In [22]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)

X_train: (121, 10)
X_valid: (41, 10)
X_test: (41, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,89,0.7355
1,train,1,cancelled,32,0.2645
2,validation,0,completed,30,0.7317
3,validation,1,cancelled,11,0.2683
4,test,0,completed,30,0.7317
5,test,1,cancelled,11,0.2683


In [23]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison

,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.634146,0.409091,0.818182,0.545455
1,Random Forest,validation,0.707317,0.428571,0.272727,0.333333
2,Dummy Most Frequent,validation,0.731707,0.000000,0.000000,0.000000


## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.732 | 0.000 | 0.000 | 0.000 |
| Logistic Regression | 0.634 | 0.409 | 0.818 | 0.545 |
| Random Forest | 0.707 | 0.429 | 0.273 | 0.333 |

![Baseline 비교](images/02.png)

- Validation에서 선택한 모델: Logistic Regression

### 결과 관찰

- Logistic Regression은 recall 0.82, F1 0.55로 Random Forest보다 accuracy는 낮지만 recall과 F1이 양호

### 나의 해석과 판단

- accuracy만 보면 Logistic Regression가 Dummy Most Frequent보다 낮으나 취소식별(recall, F1)에는 우위를 보임

### 한계와 추가 확인 사항

- 당초 목적인 취소식별에 우위를 보이는 모델사용이 적합하다고 판단


## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 파라미터 학습 | 89 | 32 |
| Validation | 모델·Threshold 선택 | 30 | 11 |
| Test | 선택 이후 최종 평가 1회 | 30 | 11 |

### 나의 해석과 판단

- 테스트를 학습, 검증하면 정답지를 미리 알려주는 개념이므로 모델 또는 Threshold 선택에 사용하면 안됨

### 교육용 random split의 한계

- 데이터가 203건으로 어떻게 나뉘느냐에 따라 결과가 달라질 수 있음

## 7. Validation 성능과 우선 지표

- Accuracy: 0.634

- Precision: 0.409

- Recall: 0.818

- F1: 0.545

### 내가 가장 중요하게 본 지표

- F1  

### 그 이유

- recall과 Precision의 균형이 중요하다 생각 (실제 FP/FN 비용 데이터가 없으므로 가정) 

In [24]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))

선택 모델: Logistic Regression
선택 임계값: 0.6


,threshold,accuracy,precision,recall,f1
8,0.60,0.804878,0.666667,0.545455,0.600000
6,0.50,0.634146,0.409091,0.818182,0.545455
7,0.55,0.707317,0.466667,0.636364,0.538462
5,0.45,0.585366,0.375000,0.818182,0.514286
4,0.40,0.560976,0.360000,0.818182,0.500000
2,0.30,0.487805,0.333333,0.909091,0.487805
3,0.35,0.536585,0.346154,0.818182,0.486486
9,0.65,0.780488,0.666667,0.363636,0.470588
1,0.25,0.365854,0.297297,1.000000,0.458333
0,0.20,0.341463,0.289474,1.000000,0.448980


## 8. Threshold 비교 

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.60 | 0.667 | 0.545 | 0.600 | 선택 |
| 0.50 | 0.409 | 0.818 | 0.545 | 기준값 |
| 0.55 | 0.467 | 0.636 | 0.538 | - |
| 0.45 | 0.375 | 0.818 | 0.514 | - |
| 0.40 | 0.360 | 0.818 | 0.500 | - |

![Threshold 비교](images/03.png)

### 내가 선택한 Threshold

- 0.60 

### 선택이유

- F1값이 제일 높음

### 나의 해석과 판단

- Threshold의 성능이 Final Test에서 바뀔 수 있으므로 Validation에서 정한 뒤 판단해야 함

In [26]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)

,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.6,frozen_before_test,0.682927,0.4,0.363636,0.380952


,pred_completed,pred_cancelled
actual_completed,24,6
actual_cancelled,7,4


,label,precision,recall,f1-score,support
0,completed,0.774194,0.800000,0.786885,30.000000
1,cancelled,0.400000,0.363636,0.380952,11.000000
2,accuracy,0.682927,0.682927,0.682927,0.682927
3,macro avg,0.587097,0.581818,0.583919,41.000000
4,weighted avg,0.673800,0.682927,0.677976,41.000000


## 9. FP/FN 해석

- FP 의미 : 실제 완료 주문을 취소로 잘못 예측 6건

- FN 의미 : 실제 취소 주문을 완료로 잘못 예측 7건

- 현재 목적에서 더 부담스러울 수 있는 오류: FN (실제 취소 건수를 맞추는 것이 목적이기 때문)

- 그 판단에 추가로 필요한 업무 정보: FP / FN 각 건의 처리금액

![Confusion Matrix](images/04.png)

### 업무분석적 의미

- 실제 취소인데 놓치게 되면 주문 비용자체가 취소되기 때문에 오류값이 더 클 것으로 판단

### 나의 해석과 판단

- FP / FN 각 건의 처리금액은 다를 수 있으므로 실제로 무엇이 더 중요한지는 비용이 있어야 판단이 될것으로 생각


## 10. Final Test

- 고정한 모델: Logistic Regression

- 고정한 Threshold: 0.60

- Accuracy: 0.683

- Precision: 0.400

- Recall: 0.364

- F1: 0.381

- Validation 대비 변화: F1 0.600 → 0.381(Precision 0.667 → 0.400, Recall 0.545 → 0.364)

### 결과 관찰

- F1 0.60이던 성능이 Test에서 0.38로 떨어짐

### 나의 해석과 판단

- 결과로 보아 Validation 대비 성능이 너무 하락하여 선택모델을 일반적으로 사용하기는 어렵다고 판단

### 선택 보호 확인

- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test로 선택을 바꾸지 않았습니다.


In [27]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()

,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,79,0,0,0.455600,Logistic Regression,0.6
1,test_0002,20,0,0,0.455474,Logistic Regression,0.6
2,test_0003,104,0,0,0.406504,Logistic Regression,0.6
3,test_0004,119,1,1,0.917927,Logistic Regression,0.6
4,test_0005,29,0,0,0.418720,Logistic Regression,0.6


In [33]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist

,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


In [31]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')

ValueError: signup_date가 order_date보다 늦은 주문이 있습니다: 45건

- 전체 파이프라인 실행 시 앞서 오류가 난 45건이 다시 문제가 됨
- LLM으로 수정 코드를 생성 재실행

In [32]:
import tempfile
from pathlib import Path

from src.classification import run_classification_analysis

# run_classification_analysis는 processed_dir의 원본 CSV를 다시 읽는다.
# 원본 data/processed에는 '가입 전 주문'이 그대로 있어, 그대로 넘기면
# build_classification_dataset의 가드(signup_date > order_date)에 걸려 ValueError가 난다.
# -> 노트북에서 이미 제외(게이트) 처리한 orders/order_items를 임시 폴더에 저장해 입력으로 사용한다.
with tempfile.TemporaryDirectory() as tmp_dir:
    gated_dir = Path(tmp_dir)
    customers.to_csv(gated_dir / 'customers_clean.csv', index=False)
    orders.to_csv(gated_dir / 'orders_clean.csv', index=False)
    order_items.to_csv(gated_dir / 'order_items_clean.csv', index=False)

    classification_result = run_classification_analysis(
        processed_dir=gated_dir,
        report_dir=REPORT_DIR,   # evidence CSV는 실제 reports 폴더에 저장
        random_state=42,
    )

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')

선택 모델: Logistic Regression
선택 임계값: 0.6


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.6,frozen_before_test,0.682927,0.4,0.363636,0.380952


model_data_internal C:\dev\llm-data-analysis-course\reports\ch10_classification_model_data_internal.csv OK
target_distribution C:\dev\llm-data-analysis-course\reports\ch10_target_distribution.csv OK
feature_audit C:\dev\llm-data-analysis-course\reports\ch10_feature_audit.csv OK
merge_checks C:\dev\llm-data-analysis-course\reports\ch10_merge_checks.csv OK
data_quality_checks C:\dev\llm-data-analysis-course\reports\ch10_data_quality_checks.csv OK
split_summary C:\dev\llm-data-analysis-course\reports\ch10_split_summary.csv OK
validation_comparison C:\dev\llm-data-analysis-course\reports\ch10_validation_model_comparison.csv OK
threshold_metrics C:\dev\llm-data-analysis-course\reports\ch10_validation_threshold_metrics.csv OK
test_metrics C:\dev\llm-data-analysis-course\reports\ch10_test_metrics.csv OK
predictions_internal C:\dev\llm-data-analysis-course\reports\ch10_classification_predictions_internal.csv OK
predictions C:\dev\llm-data-analysis-course\reports\ch10_classification_predictions

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰

- 공개 / 비공개 필요 칼럼 확인 

### 나의 해석과 판단

- 내부용은 추적을 위해 source_index, id등이 필요하나 이 부분이 공개결과 포함될 시 특정 주문,고객을 식별할 위험 있음

## 12. Validation Evidence

다음 Evidence중 확인한 항목에 체크합니다

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과

- 자동 검증 항목 전체 PASS

### 자동 PASS가 의미하지 않는 것

- 자동 검증 코드는 사전정의한 항목을 지켰다는 뜻일 뿐, 운영 적합성, 공정성, 시간에 따른 성능 변화, 모델의 성능까지 보장하지 않음

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거

1. Validation에서는 recall·F1이 개선됐으나 Final Test에서 F1이 0.38로 하락

2. recall 0.36이며, precision 0.40으로 오탐도 많음

3. Test 시 성능 격차가 커 현재 데이터로는 사용하기 어려움

### 현재 모델의 가장 큰 위험

- Validation 성능만 보고 괜찮을 것으로 판단, 사용하면 오탐건수가 많이 발생하게 됨

### 다음 개선 우선순위

1. 데이터 규모 확대

2. 실제 FP/FN 비용 확인 후 재설정

3. random split 대신 시간 순서(과거로 학습 → 미래로 평가)를 반영한 분할로 다시 검증

### 예측과 인과 구분

- 특정 주문 시점이 취소와 연관되어 보일수도 있으나 특정 결제수단/도시가 취소와 연관된다는 주장은 불가


## 최종 체크

- [x] completed=0, cancelled=1 타깃 범위를 정의했습니다.
- [x] refunded와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] line_total = quantity × unit_price를 검증했습니다. 
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다. 
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [x] 최종 Notebook URL을 제출합니다.